In [1]:
# ========================
# FINAL INTEGRATED PROTOTYPE CODE (DIRECT VIDEO CHAT - NO LOGIN, NO INTAKE)
# ========================
# Run this cell first for installs!
# %pip install ipywidgets matplotlib numpy librosa tensorflow scikit-learn transformers sounddevice SpeechRecognition pyttsx3 google-generativeai pydub deepface opencv-python # Added deepface, opencv

# --- Step 1: Imports & Setup ---
import ipywidgets as widgets
from IPython.display import display, clear_output, Audio
import matplotlib.pyplot as plt
import numpy as np
import random
import datetime
import librosa
import librosa.display
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, LSTM, Dropout
# Removed unused imports: RandomForestClassifier, Sequential, Conv2D, MaxPooling2D, Flatten (if only using pre-trained models)
from sklearn.preprocessing import OneHotEncoder
from transformers import pipeline
import sounddevice as sd
import speech_recognition as sr
import pyttsx3
import google.generativeai as genai
import threading
import queue
import time
import os
from pydub import AudioSegment
import io
import logging
import cv2
from deepface import DeepFace
import sys
# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# For reproducibility and silencing TF/DeepFace warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')
np.random.seed(42)
tf.random.set_seed(42)

print("Libraries loaded.")

# --- Global Config & Resources ---
AUDIO_SAMPLE_RATE = 16000; AUDIO_CHANNELS = 1; AUDIO_CHUNK_DURATION = 0.5
AUDIO_CHUNK_SAMPLES = int(AUDIO_SAMPLE_RATE * AUDIO_CHUNK_DURATION)
STT_PHRASE_TIME_LIMIT = 7; VIDEO_DETECTION_INTERVAL = 5
DEEPFACE_DETECTOR_BACKEND = 'opencv'

# Queues & Events
stt_text_q = queue.Queue(); tts_q = queue.Queue(); audio_chunk_q = queue.Queue(maxsize=20)
stop_event = threading.Event(); listen_event = threading.Event()

# Shared State - Simplified session data (no intake/issues)
session_data = {
    "chat_log": [],
    "voice_emotion_log": [],
    "face_emotion_log": []
}
# Locks for logs
voice_emotion_log_lock = threading.Lock()
face_emotion_log_lock = threading.Lock()
chat_history_lock = threading.Lock() # Still need for chat history list modifications

# Global emotion holders
last_detected_voice_emotion_global = "N/A"; last_detected_face_emotion_global = "N/A"

# Plotting & Widget variables
ser_fig, ser_ax = None, None
spectrogram_output_widget = None; video_display_widget = None
audio_stream = None; video_capture = None
main_area = widgets.Output() # Main display area for the chat UI

print("Shared resources initialized.")

D:\anaconda\envs\tf_env\lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)



Libraries loaded.
Shared resources initialized.


In [5]:
import os
import sys
import time
import random
import queue
import logging
import datetime
import threading

import numpy as np
import librosa
import librosa.display
import cv2
import matplotlib.pyplot as plt
from matplotlib import rcParams

from tensorflow.keras.models import load_model
from sklearn.preprocessing import OneHotEncoder

import sounddevice as sd
import speech_recognition as sr
import pyttsx3
import ipywidgets as widgets
from IPython.display import display, clear_output

# Import DeepFace for face emotion analysis
from deepface import DeepFace

# Optional: Import or configure your LLM if available
# import genai

# -----------------------------
# Constants and Global Variables
# -----------------------------
AUDIO_SAMPLE_RATE = 16000  # Adjust as needed
AUDIO_CHANNELS = 1
AUDIO_CHUNK_SAMPLES = 2048

VIDEO_DETECTION_INTERVAL = 30
DEEPFACE_DETECTOR_BACKEND = "opencv"  # or another backend that you have installed

# Global variables for threading and state
stop_event = threading.Event()
listen_event = threading.Event()

audio_chunk_q = queue.Queue(maxsize=10)
stt_text_q = queue.Queue(maxsize=10)
tts_q = queue.Queue(maxsize=10)

chat_history_lock = threading.Lock()
face_emotion_log_lock = threading.Lock()
voice_emotion_log_lock = threading.Lock()

session_data = {"chat_log": [], "voice_emotion_log": [], "face_emotion_log": []}

last_detected_voice_emotion_global = "N/A"
last_detected_face_emotion_global = "N/A"
audio_stream = None
video_capture = None
ser_fig = None
ser_ax = None
spectrogram_output_widget = None
video_display_widget = None
main_area = widgets.Output()

# -----------------------------
# Step 5: Load Models
# -----------------------------
FACE_EMOTIONS = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]
SER_MODEL_PATH = 'speech_emotion_model.h5'
ser_model = None
ser_encoder = None
SER_EMOTIONS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'ps', 'sad']

try:
    ser_encoder = OneHotEncoder()
    ser_encoder.fit(np.array(SER_EMOTIONS).reshape(-1, 1))
    ser_model = load_model(SER_MODEL_PATH)
    logging.info(f"Loaded SER model from {SER_MODEL_PATH}")
except Exception as e:
    logging.error(f"ERROR loading SER model from {SER_MODEL_PATH}: {e} - Voice Emotion Disabled.")
    ser_model = None

def analyze_voice_emotion_keras(audio_chunk):
    if ser_model is None or ser_encoder is None:
        return "SER N/A", 0.0
    emotion, confidence = "unknown", 0.0
    try:
        if audio_chunk.dtype != np.float32:
            audio_chunk = audio_chunk.astype(np.float32) / 32767.0
        if len(audio_chunk) < AUDIO_CHUNK_SAMPLES:
            return "Too short", 0.0
        mfcc = np.mean(librosa.feature.mfcc(y=audio_chunk, sr=AUDIO_SAMPLE_RATE, n_mfcc=40).T, axis=0)
        mfcc_reshaped = np.expand_dims(np.expand_dims(mfcc, axis=0), axis=-1)
        prediction = ser_model.predict(mfcc_reshaped, verbose=0)
        emotion_index = np.argmax(prediction[0])
        confidence = prediction[0][emotion_index]
        emotion = ser_encoder.inverse_transform(np.array(emotion_index).reshape(1, -1))[0][0]
    except Exception as e:
        logging.warning(f"SER Predict Err: {e}")
        emotion = "SER Error"
    return emotion, confidence

logging.info("Loading NLP pipeline...")
nlp_text_analyzer = None
try:
    # Using Hugging Face pipeline for sentiment analysis
    from transformers import pipeline
    nlp_text_analyzer = pipeline("sentiment-analysis")
    logging.info("NLP pipeline loaded.")
except Exception as e:
    logging.warning(f"NLP pipeline load failed: {e}.")

def analyze_text_nlp_hf(text):
    if nlp_text_analyzer:
        try:
            result = nlp_text_analyzer(text)[0]
            return result["label"], result["score"]
        except Exception as e:
            logging.warning(f"NLP Err: {e}")
            return "NLP Error", 0.0
    else:
        return "NEUTRAL", 0.5

logging.info("Configuring LLM...")
llm_model = None
try:
    GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY')
    if GOOGLE_API_KEY:
        # Configure your LLM model (replace with your actual configuration)
        # genai.configure(api_key=GOOGLE_API_KEY)
        # llm_model = genai.GenerativeModel('gemini-pro')
        logging.info("Gemini configured.")
    else:
        logging.warning("Warn: GOOGLE_API_KEY missing. LLM disabled.")
except Exception as e:
    logging.error(f"LLM Config Err: {e}. LLM disabled.")

logging.info("Initializing TTS engine...")
tts_engine = None
try:
    tts_engine = pyttsx3.init()
    logging.info("TTS initialized.")
except Exception as e:
    logging.warning(f"TTS init failed: {e}.")

# -----------------------------
# Step 6: Real-time Processing Threads & Functions
# -----------------------------
def audio_callback(indata, frames, time_info, status):
    if status:
        print("Audio Status:", status, file=sys.stderr)
    if not stop_event.is_set():
        try:
            audio_chunk_q.put(indata.copy(), timeout=0.05)
        except queue.Full:
            pass

def audio_processing_thread_ser():
    global ser_fig, ser_ax, last_detected_voice_emotion_global, session_data
    logging.info("Audio SER/Spectrogram thread started.")
    while not stop_event.is_set():
        try:
            audio_chunk = audio_chunk_q.get(timeout=0.5)
            if audio_chunk.ndim > 1:
                audio_chunk = audio_chunk.flatten()
            voice_emotion, ser_confidence = analyze_voice_emotion_keras(audio_chunk)
            last_detected_voice_emotion_global = voice_emotion
            if voice_emotion not in ["SER N/A", "Too short", "SER Error"]:
                timestamp = datetime.datetime.now().strftime('%H:%M:%S.%f')[:-3]
                with voice_emotion_log_lock:
                    session_data["voice_emotion_log"].append((timestamp, voice_emotion, ser_confidence))
            if ser_fig is not None and ser_ax is not None and spectrogram_output_widget is not None and spectrogram_output_widget.layout.display != "none":
                try:
                    S = librosa.feature.melspectrogram(y=audio_chunk.astype(np.float32), sr=AUDIO_SAMPLE_RATE, n_mels=128, fmax=8000)
                    S_dB = librosa.power_to_db(S, ref=np.max)
                    ser_ax.clear()
                    librosa.display.specshow(S_dB, sr=AUDIO_SAMPLE_RATE, x_axis='time', y_axis='mel', ax=ser_ax, fmax=8000)
                    ser_ax.set_title(f"Spectrogram (Voice: {voice_emotion})")
                    with spectrogram_output_widget:
                        clear_output(wait=True)
                        display(ser_fig)
                except Exception as plot_e:
                    pass
            audio_chunk_q.task_done()
        except queue.Empty:
            continue
        except Exception as thread_e:
            logging.error(f"SER Thread Err: {thread_e}")
            time.sleep(0.1)
    logging.info("Audio SER/Spectrogram thread finished.")

def video_capture_face_emotion_thread():
    global last_detected_face_emotion_global, video_capture, video_display_widget, session_data
    logging.info("Video/Face thread started.")
    frame_count = 0
    last_analysis = {}
    try:
        video_capture = cv2.VideoCapture(0)
        assert video_capture.isOpened(), "Webcam Error"
    except Exception as e:
        logging.error(f"Webcam Init Error: {e}")
        stop_event.set()
        return
    logging.info("Webcam opened.")
    while not stop_event.is_set():
        ret, frame = video_capture.read()
        if not ret:
            time.sleep(0.1)
            continue
        display_frame = frame.copy()
        frame_count += 1
        if frame_count % VIDEO_DETECTION_INTERVAL == 0:
            face_emotion, confidence, face_region = "N/A", 0.0, None
            try:
                results = DeepFace.analyze(img_path=frame, actions=['emotion'],
                                           detector_backend=DEEPFACE_DETECTOR_BACKEND,
                                           enforce_detection=False, silent=True)
                if results and isinstance(results, list) and results[0]:
                    analysis = results[0]
                    face_emotion = analysis.get('dominant_emotion', "N/A")
                    confidence = analysis.get('emotion', {}).get(face_emotion, 0.0)
                    face_region = analysis.get('region')
                    last_analysis = {'emotion': face_emotion, 'confidence': confidence, 'region': face_region}
                    timestamp = datetime.datetime.now().strftime('%H:%M:%S.%f')[:-3]
                    with face_emotion_log_lock:
                        session_data["face_emotion_log"].append((timestamp, face_emotion, confidence))
                else:
                    last_analysis = {}
                    face_emotion = "N/A"
                last_detected_face_emotion_global = face_emotion
            except Exception as e:
                last_detected_face_emotion_global = "FER Error"
                last_analysis = {}
                logging.warning(f"DeepFace Error: {e}")
        if 'region' in last_analysis and last_analysis['region']:
            r = last_analysis['region']
            cv2.rectangle(display_frame, (r['x'], r['y']), (r['x']+r['w'], r['y']+r['h']), (0, 255, 0), 1)
            emo = last_analysis.get('emotion', '?')
            conf = last_analysis.get('confidence', 0)
            cv2.putText(display_frame, f"{emo} ({conf*100:.0f}%)", (r['x'], r['y']-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)
        if video_display_widget is not None:
            try:
                _, encoded_image = cv2.imencode('.jpg', display_frame, [cv2.IMWRITE_JPEG_QUALITY, 80])
                video_display_widget.value = encoded_image.tobytes()
            except Exception as e:
                pass
        time.sleep(0.03)
    logging.info("Video/Face thread finished.")
    if video_capture:
        video_capture.release()
        video_capture = None

stt_recognizer = sr.Recognizer()

def speech_to_text_thread():
    logging.info("STT thread started.")
    mic = sr.Microphone(sample_rate=AUDIO_SAMPLE_RATE)
    with mic as source:
        logging.info("STT Adjust Noise...")
        time.sleep(0.5)
        try:
            stt_recognizer.adjust_for_ambient_noise(source, duration=1.0)
            logging.info(f"STT Thresh {stt_recognizer.energy_threshold:.1f}")
        except Exception as e:
            logging.warning(f"STT Noise Adjust Fail: {e}")
    while not stop_event.is_set():
        if listen_event.wait(timeout=1.0):
            logging.info("STT Listening...")
            text = "[STT No Trigger]"
            try:
                with mic as source:
                    audio_data = stt_recognizer.listen(source, phrase_time_limit=5, timeout=6)
                logging.info("STT Processing...")
                text = stt_recognizer.recognize_google(audio_data)
            except sr.WaitTimeoutError:
                text = "[STT Timeout]"
                logging.info(text)
            except sr.UnknownValueError:
                text = "[STT Audio Not Understood]"
                logging.info(text)
            except sr.RequestError as e:
                text = "[STT API Error]"
                logging.warning(text)
            except Exception as e:
                text = "[STT Sys Error]"
                logging.error(f"STT Error: {e}", exc_info=True)
            finally:
                stt_text_q.put(text)
                listen_event.clear()
    logging.info("STT thread finished.")

def text_to_speech_thread():
    logging.info("TTS thread started.")
    while not stop_event.is_set():
        try:
            text = tts_q.get(timeout=0.5)
        except queue.Empty:
            continue
        try:
            if tts_engine:
                tts_engine.say(text)
                tts_engine.runAndWait()
            tts_q.task_done()
        except Exception as e:
            logging.error(f"TTS Error: {e}")
            tts_q.task_done()
    logging.info("TTS thread finished.")

# -----------------------------
# Step 7: Chatbot Logic Function
# -----------------------------
def get_chatbot_reply_final(user_text, current_voice_emotion, current_facial_emotion):
    global session_data
    bot_response = "..."
    if user_text.startswith("[STT") and user_text.endswith("]"):
        if "Not Understood" in user_text:
            return random.choice(["Could you repeat that?", "Sorry?"])
        elif "Timeout" in user_text:
            return random.choice(["Didn't hear anything.", "Listening again."])
        else:
            return "Speech input issue."
    text_sentiment, _ = analyze_text_nlp_hf(user_text)
    with chat_history_lock:
        session_data["chat_log"].append({'role': 'user', 'parts': [user_text]})
        limited_history = session_data["chat_log"][-10:]
    if llm_model:
        try:
            context_prompt = (f"AI Assistant Prompt: Empathetic & concise. Gently suggest crisis line (988)/professional help if severe distress/self-harm is mentioned.\n"
                              f"Ctx: V-Emo={current_voice_emotion}, F-Emo={current_facial_emotion}, T-Sent={text_sentiment}\nHistory:")
            gemini_content = [{'role': 'user', 'parts': [context_prompt]},
                              {'role': 'model', 'parts': ["Okay."]}] + limited_history
            response = llm_model.generate_content(gemini_content, stream=False,
                                                  request_options={'timeout': 45},
                                                  safety_settings={'HARASSMENT':'block_none',
                                                                   'HATE_SPEECH':'block_none',
                                                                   'SEXUALLY_EXPLICIT':'block_none',
                                                                   'DANGEROUS_CONTENT':'block_none'})
            if response.parts:
                bot_response = response.text
            else:
                logging.warning(f"LLM Blocked: {response.prompt_feedback}")
                bot_response = "Let's change the subject."
        except Exception as e:
            logging.error(f"LLM Error: {e}")
            bot_response = "My thinking cap fell off. Try again?"
    else:
        combined = f"V:{current_voice_emotion}, F:{current_facial_emotion}"
        bot_response = f"Okay. ({combined}). Please continue." if "hello" not in user_text.lower() else f"Hello! ({combined})"
    with chat_history_lock:
        session_data["chat_log"].append({'role': 'model', 'parts': [bot_response]})
        if len(session_data["chat_log"]) > 40:
            session_data["chat_log"] = session_data["chat_log"][-10:]
    if "goodbye quit now" in user_text.lower():
        bot_response = "Okay, goodbye! Ending the session."
        if tts_engine:
            tts_q.put(bot_response)
            tts_q.join()
        stop_event.set()
    return bot_response

# -----------------------------
# Step 8: Chat Interface Logic (Video Chat)
# -----------------------------
def start_video_chat_directly():
    global audio_stream, video_capture, ser_fig, ser_ax
    global spectrogram_output_widget, video_display_widget, session_data

    logging.info("Initializing Video Chat Session UI and threads...")
    with main_area:
        clear_output(wait=True)
        print("--- Video Chat Session Active ---")

    session_data["chat_log"] = []
    session_data["voice_emotion_log"] = []
    session_data["face_emotion_log"] = []

    status_label_vc = widgets.Label(value="Status: Initializing...")
    video_display_widget = widgets.Image(format='jpeg', width=480)
    video_box = widgets.VBox([widgets.Label("Live Video:"), video_display_widget],
                             layout={'border': '1px solid grey', 'width':'500px'})
    spectrogram_output_widget = widgets.Output(layout={'border': '1px solid #ccc', 'width':'400px'})
    spectrogram_box = widgets.VBox([widgets.Label("Live Spectrogram:"), spectrogram_output_widget],
                                   layout={'margin_top': '0px', 'width':'410px'})
    ser_fig, ser_ax = plt.subplots(figsize=(5, 3))
    plt.close(ser_fig)
    ser_ax.set_title("Spectrogram...")
    with spectrogram_output_widget:
        display(ser_fig)
    chat_log_output_vc = main_area
    listen_button_vc = widgets.Button(description="🎤 Speak", button_style='info', icon='microphone')
    end_chat_button_vc = widgets.Button(description="End & Report", button_style='danger')
    controls_box = widgets.HBox([listen_button_vc, end_chat_button_vc], layout={'margin_top': '10px'})
    top_row = widgets.HBox([video_box, spectrogram_box])
    video_chat_main_box = widgets.VBox([status_label_vc, top_row, controls_box])
    
    with main_area:
        display(video_chat_main_box)
    
    status_label_vc.value = "Status: Starting threads..."
    logging.info("Starting threads...")
    stop_event.clear()

    threads = []
    vid_thread = threading.Thread(target=video_capture_face_emotion_thread, daemon=True)
    threads.append(vid_thread)
    vid_thread.start()
    time.sleep(0.5)
    stt_thread = threading.Thread(target=speech_to_text_thread, daemon=True)
    threads.append(stt_thread)
    stt_thread.start()
    ser_thread = threading.Thread(target=audio_processing_thread_ser, daemon=True)
    threads.append(ser_thread)
    ser_thread.start()
    if tts_engine:
        tts_thread = threading.Thread(target=text_to_speech_thread, daemon=True)
        threads.append(tts_thread)
        tts_thread.start()
    logging.info("Threads started.")

    try:
        status_label_vc.value = "Status: Starting audio..."
        audio_stream = sd.InputStream(samplerate=AUDIO_SAMPLE_RATE,
                                      channels=AUDIO_CHANNELS,
                                      blocksize=AUDIO_CHUNK_SAMPLES,
                                      callback=audio_callback,
                                      dtype='float32')
        audio_stream.start()
        status_label_vc.value = "Status: Ready."
        logging.info("Audio stream started.")
        with chat_log_output_vc:
            initial_msg = "Bot: Video chat active. How can I help you today?"
            print(initial_msg)
        if tts_engine:
            tts_q.put(initial_msg)
        with chat_history_lock:
            session_data["chat_log"].append({'role': 'model', 'parts': [initial_msg]})
    except Exception as e:
        status_label_vc.value = "Status: AUDIO ERROR"
        logging.error(f"Audio Stream Error: {e}")
        stop_event.set()
        return

    def on_listen_vc_clicked(b):
        listen_event.set()
        status_label_vc.value = "Status: Listening..."
    def on_end_vchat_clicked(b):
        status_label_vc.value = "Status: Ending..."
        logging.info("End button clicked.")
        stop_event.set()
        listen_button_vc.disabled = True
        end_chat_button_vc.disabled = True

    listen_button_vc.on_click(on_listen_vc_clicked)
    end_chat_button_vc.on_click(on_end_vchat_clicked)
    logging.info("Video chat setup complete.")

# -----------------------------
# Step 9: Session Report Generation
# -----------------------------
def generate_report():
    global session_data
    logging.info("Generating session report.")
    report_lines = []
    username = "user_session"
    report_lines.append("="*40)
    report_lines.append("  Multi-Modal Chatbot Session Report")
    report_lines.append("="*40)
    report_lines.append(f"Date: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

    report_lines.append("--- Voice Emotion Log ---")
    log_v = session_data.get("voice_emotion_log", [])
    if log_v:
        report_lines.append("  Timestamp   | Emotion         | Confidence")
        report_lines.append("  ------------|-----------------|-----------")
        with voice_emotion_log_lock:
            log_limit = 30
            display_log = log_v[-log_limit:]
            if len(log_v) > log_limit:
                report_lines.append(f"  ...(last {log_limit})...")
            report_lines.extend([f"  {ts} | {emo:<15} | {conf:.2f}" for ts, emo, conf in display_log])
    else:
        report_lines.append("  (No voice emotions logged)")

    report_lines.append("\n--- Face Emotion Log ---")
    log_f = session_data.get("face_emotion_log", [])
    if log_f:
        report_lines.append("  Timestamp   | Emotion         | Confidence")
        report_lines.append("  ------------|-----------------|-----------")
        with face_emotion_log_lock:
            log_limit = 30
            display_log = log_f[-log_limit:]
            if len(log_f) > log_limit:
                report_lines.append(f"  ...(last {log_limit})...")
            report_lines.extend([f"  {ts} | {emo:<15} | {conf:.2f}" for ts, emo, conf in display_log])
    else:
        report_lines.append("  (No face emotions logged)")

    report_lines.append("\n--- Conversation Transcript ---")
    hist = session_data.get("chat_log", [])
    if hist:
        for t in hist:
            role = t.get('role', '?').capitalize()
            message = t.get('parts', [''])[0]
            report_lines.append(f"  {role}: {message}")
    else:
        report_lines.append("  (No conversation recorded)")

    report_lines.append("\n" + "="*40)
    report_content = "\n".join(report_lines)
    filename = f"{username}_report_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    try:
        with open(filename, "w", encoding='utf-8') as f:
            f.write(report_content)
        logging.info(f"Report generated: '{filename}'")
        print(f"\nReport generated: '{filename}'")
    except Exception as e:
        logging.error(f"Report write Err: {e}")
        print(f"\nReport write Error: {e}")

    print("\nRemember: This is a prototype and not a substitute for professional advice.")

# -----------------------------
# End Chat Procedure
# -----------------------------
def end_chat_procedure():
    logging.info("Ending chat session procedure...")
    if not stop_event.is_set():
        stop_event.set()
        logging.info("Stop event set by end procedure.")

    global audio_stream, video_capture
    if audio_stream:
        try:
            logging.info("Stopping audio...")
            audio_stream.stop()
            audio_stream.close()
            audio_stream = None
            logging.info("Audio stopped.")
        except Exception as e:
            logging.error(f"Audio Stop Err: {e}")
    if video_capture:
        try:
            logging.info("Releasing video...")
            video_capture.release()
            video_capture = None
            logging.info("Video released.")
        except Exception as e:
            logging.error(f"Video Release Err: {e}")
    try:
        cv2.destroyAllWindows()
        [cv2.waitKey(1) for _ in range(5)]
    except Exception as cv_e:
        logging.warning(f"OpenCV cleanup warn: {cv_e}")

    logging.info("Waiting briefly for threads to finish...")
    time.sleep(2.0)
    with main_area:
        clear_output(wait=True)
        generate_report()
    print("\nSession ended. Run the cell again to restart the chatbot.")

    global session_data, last_detected_face_emotion_global, last_detected_voice_emotion_global
    session_data = {"chat_log": [], "voice_emotion_log": [], "face_emotion_log": []}
    last_detected_face_emotion_global = "N/A"
    last_detected_voice_emotion_global = "N/A"

# -----------------------------
# Step 10: Main Application Execution Logic
# -----------------------------
def run_prototype():
    global main_area, session_data

    with main_area:
        clear_output()
    print("="*50)
    print(" Multi-Modal Mental Health Chatbot Prototype")
    print("(Direct Video Chat)")
    print("="*50)
    display(main_area)

    logging.info("Prototype starting, initiating video chat.")
    start_video_chat_directly()

    try:
        logging.info("Main interaction loop active. Monitoring speech queue.")
        while not stop_event.is_set():
            try:
                user_text = stt_text_q.get(timeout=0.5)
                current_chat_log_widget = main_area
                if user_text:
                    logging.info(f"STT Received: {user_text}")
                    with current_chat_log_widget:
                        if not user_text.startswith("[STT"):
                            print(f"\n>> You: {user_text}")
                    if not user_text.startswith("[STT"):
                        current_voice = last_detected_voice_emotion_global
                        current_face = last_detected_face_emotion_global
                        bot_reply = get_chatbot_reply_final(user_text, current_voice, current_face)
                        with current_chat_log_widget:
                            print(f">> Bot: {bot_reply}")
                        if tts_engine and bot_reply:
                            tts_q.put(bot_reply)
                stt_text_q.task_done()
                if stop_event.is_set():
                    logging.info("Stop event set by bot response, breaking loop.")
                    break
            except queue.Empty:
                time.sleep(0.1)
                continue
            except Exception as main_loop_e:
                logging.error(f"!! Main Loop Error: {main_loop_e} !!", exc_info=True)
                time.sleep(1)
    except KeyboardInterrupt:
        logging.info("Keyboard Interrupt! Stopping...")
        stop_event.set()
    finally:
        logging.info("Main interaction loop exited.")
        if stop_event.is_set():
            end_chat_procedure()
        logging.info("Prototype Shutdown Complete.")

# -----------------------------
# Start the Prototype
# -----------------------------
run_prototype()


2025-04-03 14:18:26,534 - WARNING - Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.
2025-04-03 14:18:26,544 - INFO - Loaded SER model from speech_emotion_model.h5
2025-04-03 14:18:26,544 - INFO - Loading NLP pipeline...
No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu
2025-04-03 14:18:27,801 - INFO - NLP pipeline loaded.
2025-04-03 14:18:27,801 - INFO - Configuring LLM...
2025-04-03 14:18:27,803 - WARNING - Warn: GOOGLE_API_KEY missing. LLM disabled.
2025-04-03 14:18:27,803 - INFO - Initializing TTS engine...
2025-04-03 14:18:27,926 - INFO - Imported existing <module 'comtypes.gen' from 'D:\\anaconda\\envs\\tf_env\\lib\

 Multi-Modal Mental Health Chatbot Prototype
(Direct Video Chat)


Output()

2025-04-03 14:18:29,073 - INFO - Prototype starting, initiating video chat.
2025-04-03 14:18:29,073 - INFO - Initializing Video Chat Session UI and threads...
2025-04-03 14:18:29,375 - INFO - Starting threads...
2025-04-03 14:18:29,381 - INFO - Video/Face thread started.
2025-04-03 14:18:29,885 - INFO - STT thread started.
2025-04-03 14:18:29,885 - INFO - Audio SER/Spectrogram thread started.
2025-04-03 14:18:29,894 - INFO - TTS thread started.
2025-04-03 14:18:29,894 - INFO - Threads started.
2025-04-03 14:18:30,183 - INFO - Webcam opened.
2025-04-03 14:18:30,814 - INFO - Audio stream started.
2025-04-03 14:18:30,818 - INFO - Video chat setup complete.
2025-04-03 14:18:30,819 - INFO - Main interaction loop active. Monitoring speech queue.
2025-04-03 14:18:30,839 - INFO - STT Adjust Noise...
2025-04-03 14:18:32,154 - INFO - STT Thresh 4460.9
2025-04-03 14:18:39,173 - WARNING - SER Predict Err: Shape of the passed X data is not correct. Expected 7 columns, got 1.
2025-04-03 14:18:39,465


Session ended. Run the cell again to restart the chatbot.
